In [ ]:
!pip install kagglehub pandas gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 61.9 MB/s eta 0:00:00


In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("scodepy/customer-support-intent-dataset")
print("Path:", path)

print(os.listdir(path))

100%|██████████| 60.0k/60.0k [00:00<00:00, 48.1MB/s]

Extracting files...
Path: /root/.cache/kagglehub/datasets/scodepy/customer-support-intent-dataset/versions/1
['Bitext_Sample_Customer_Service_Testing_Dataset.csv', 'Bitext_Sample_Customer_Service_Training_Dataset.csv', 'Bitext_Sample_Customer_Service_Validation_Dataset.csv']


In [ ]:
import pandas as pd
import os

# File paths
train_path = os.path.join(path, "Bitext_Sample_Customer_Service_Training_Dataset.csv")
test_path = os.path.join(path, "Bitext_Sample_Customer_Service_Testing_Dataset.csv")
val_path = os.path.join(path, "Bitext_Sample_Customer_Service_Validation_Dataset.csv")

# Load data
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
val_df = pd.read_csv(val_path)

train_df.head()

,utterance,intent,category,tags
0,would it be possible to cancel the order I made?,cancel_order,ORDER,BIP
1,cancelling order,cancel_order,ORDER,BK
2,I need assistance canceling the last order I h...,cancel_order,ORDER,B
3,problem with canceling the order I made,cancel_order,ORDER,B
4,I don't know how to cancel the order I made,cancel_order,ORDER,B


In [ ]:
print(train_df.columns)

Index(['utterance', 'intent', 'category', 'tags'], dtype='object')


In [ ]:

train_df = train_df.rename(columns={"utterance": "query", "intent": "label"})
test_df = test_df.rename(columns={"utterance": "query", "intent": "label"})
val_df = val_df.rename(columns={"utterance": "query", "intent": "label"})


df = pd.concat([train_df, test_df, val_df])

df.head()

,query,label,category,tags
0,would it be possible to cancel the order I made?,cancel_order,ORDER,BIP
1,cancelling order,cancel_order,ORDER,BK
2,I need assistance canceling the last order I h...,cancel_order,ORDER,B
3,problem with canceling the order I made,cancel_order,ORDER,B
4,I don't know how to cancel the order I made,cancel_order,ORDER,B


In [ ]:
import string

def preprocess(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

texts = df["query"].tolist()
labels = df["label"].tolist()

processed_texts = [preprocess(t) for t in texts]

In [ ]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(sentences=processed_texts, vector_size=50, window=5, min_count=1)

In [ ]:
import numpy as np

def get_sentence_vector(sentence):
    words = preprocess(sentence)
    vectors = []

    for word in words:
        if word in w2v_model.wv:
            vectors.append(w2v_model.wv[word])

    if len(vectors) == 0:
        return np.zeros(50)

    return np.mean(vectors, axis=0)

X = np.array([get_sentence_vector(t) for t in texts])
y = np.array(labels)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.svm import SVC

model = SVC(kernel='linear')
model.fit(X_train, y_train)

SVC(kernel='linear')

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9229357798165138

Classification Report:

                          precision    recall  f1-score   support

            cancel_order       0.89      0.88      0.88        64
            change_order       0.83      0.70      0.76        57
 change_shipping_address       0.94      0.97      0.95        65
  check_cancellation_fee       0.93      1.00      0.97        56
           check_invoice       0.96      0.99      0.97        76
   check_payment_methods       0.98      1.00      0.99        57
     check_refund_policy       1.00      0.95      0.98        63
               complaint       1.00      0.99      0.99        72
contact_customer_service       0.96      0.96      0.96        57
     contact_human_agent       0.91      0.96      0.94        53
          create_account       0.73      0.87      0.79        60
          delete_account       0.73      0.70      0.71        57
        delivery_options       0.98      0.97      0.98        67
         delivery_per

In [ ]:
def predict_query(query):
    vec = get_sentence_vector(query).reshape(1, -1)
    return model.predict(vec)[0]

In [ ]:
print(predict_query("I forgot my password"))
print(predict_query("My payment failed"))
print(predict_query("Account is hacked."))

recover_password
payment_issue
delete_account
